In [1]:
# Cell 1: Setup & Constants
# Notebook 04: Fact_TransactionPremium — Gold_SalesOps_Fact_TransactionPremium
# Source: ClientUnderwriterPremium (Silver) + Transaction (ClientPartyId) + Policy (InceptionDate)
# Two LEFT JOINs. PolicyId is unique in Policy. TransactionId join for client resolution.
# Set RUN_STATS = False for fast production runs, True for debugging with full stats.

from pyspark.sql import functions as F

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

SILVER_BASE = "abfss://CRBDATA_PAS_EM20_PRE@onelake.dfs.fabric.microsoft.com/PAS_U_EM20_LH_Silver.Lakehouse/Tables/dbo"
RUN_STATS = False

print("Setup complete.")
print(f"Silver base: {SILVER_BASE}")
print(f"RUN_STATS: {RUN_STATS}")

StatementMeta(, f1b3efe3-c90b-45f6-9d62-00a7d2401418, 3, Finished, Available, Finished, False)

Setup complete.
Silver base: abfss://CRBDATA_PAS_EM20_PRE@onelake.dfs.fabric.microsoft.com/PAS_U_EM20_LH_Silver.Lakehouse/Tables/dbo
RUN_STATS: False


In [2]:
# Cell 2: Load ClientUnderwriterPremium

df_cup = (
    spark.read.format("delta").load(f"{SILVER_BASE}/ClientUnderwriterPremium")
    .filter(F.col("IsDeleted") == False)
    .filter(F.col("PolicyId") != -1)
    .filter(F.col("PolicyId").isNotNull())
)

if RUN_STATS:
    cup_count = df_cup.count()
    print(f"ClientUnderwriterPremium rows: {cup_count:,}\n")

    print("Key column NULL counts:")
    print("-" * 55)
    for col_name in ["PolicyId", "PolicySectionId", "TransactionId", "UWGlobalPartyId", "UnderwriterId", "USDExchangeRate"]:
        null_count = df_cup.filter(F.col(col_name).isNull()).count()
        pct = null_count / cup_count * 100 if cup_count > 0 else 0
        print(f"  {col_name:<35} {null_count:>12,}  ({pct:5.1f}%)")

    display(df_cup.limit(5))
else:
    print("ClientUnderwriterPremium loaded.")

StatementMeta(, f1b3efe3-c90b-45f6-9d62-00a7d2401418, 4, Finished, Available, Finished, False)

ClientUnderwriterPremium loaded.


In [3]:
# Cell 3: Load Transaction table (for InsuredPartyId / ClientPartyId + OwnershipOrganisation)

df_txn = (
    spark.read.format("delta").load(f"{SILVER_BASE}/Transaction")
    .filter(F.col("IsDeleted") == False)
    .select("TransactionId", "InsuredPartyId", "ClientPartyId", "OwnershipOrganisationId",
            "ProductKey",
            "GlobalProductClassId", "GlobalProductClass",
            "GlobalProductLineId", "GlobalProductLine",
            "GlobalProductId", "GlobalProduct")
)

if RUN_STATS:
    txn_count = df_txn.count()
    print(f"Transaction rows: {txn_count:,}")
else:
    print("Transaction loaded.")

StatementMeta(, f1b3efe3-c90b-45f6-9d62-00a7d2401418, 5, Finished, Available, Finished, False)

Transaction loaded.


In [4]:
# Cell 4: Load Policy table (for InceptionDate, ExpiryDate, RenewalDate)

df_policy = (
    spark.read.format("delta").load(f"{SILVER_BASE}/Policy")
    .filter(F.col("IsDeleted") == False)
    .select("PolicyId", "InceptionDate", "FirstInceptionDate", "ExpiryDate", "RenewalDate", "RefInsuranceTypeId", "RefInsuranceType")
)

if RUN_STATS:
    policy_count = df_policy.count()
    policy_distinct = df_policy.select("PolicyId").distinct().count()
    print(f"Policy rows:          {policy_count:,}")
    print(f"Distinct PolicyId:    {policy_distinct:,}")
    print(f"PolicyId unique?      {'YES' if policy_distinct == policy_count else 'NO'}")
else:
    print("Policy loaded.")

StatementMeta(, f1b3efe3-c90b-45f6-9d62-00a7d2401418, 6, Finished, Available, Finished, False)

Policy loaded.


In [5]:
# Cell 5: Join CUP + Transaction on TransactionId → get ClientPartyId

before_count = df_cup.count()

df_joined = df_cup.join(df_txn, on="TransactionId", how="left")

# Resolve client: InsuredPartyId preferred, fallback to ClientPartyId
df_joined = df_joined.withColumn(
    "ClientPartyId",
    F.coalesce(F.col("InsuredPartyId"), F.col("ClientPartyId"))
).drop("InsuredPartyId")

after_count = df_joined.count()

# DUPLICATE CHECK — always runs
print(f"Row count BEFORE Transaction join: {before_count:,}")
print(f"Row count AFTER Transaction join:  {after_count:,}")
print(f"STATUS: {'No duplicates' if before_count == after_count else f'DUPLICATES DETECTED ({after_count - before_count:,} extra rows)'}")

StatementMeta(, f1b3efe3-c90b-45f6-9d62-00a7d2401418, 7, Finished, Available, Finished, False)

Row count BEFORE Transaction join: 48,159,303
Row count AFTER Transaction join:  48,159,303
STATUS: No duplicates


In [6]:
# Cell 6: Join + Policy on PolicyId → get InceptionDate, ExpiryDate

before_count = after_count

df_joined = df_joined.join(df_policy, on="PolicyId", how="left")

after_count = df_joined.count()

# DUPLICATE CHECK — always runs
print(f"Row count BEFORE Policy join: {before_count:,}")
print(f"Row count AFTER Policy join:  {after_count:,}")
print(f"STATUS: {'No duplicates' if before_count == after_count else f'DUPLICATES DETECTED ({after_count - before_count:,} extra rows)'}")

StatementMeta(, f1b3efe3-c90b-45f6-9d62-00a7d2401418, 8, Finished, Available, Finished, False)

Row count BEFORE Policy join: 48,159,303
Row count AFTER Policy join:  48,159,303
STATUS: No duplicates


In [7]:
# Cell 6b: Load Organisation + Join on OwnershipOrganisationId

df_org = (
    spark.read.format("delta").load(f"{SILVER_BASE}/Organisation")
    .filter(F.col("IsDeleted") == False)
    .select(
        F.col("OrganisationId"),
        F.col("Organisation"),
    )
)

before_count = df_joined.count()

df_joined = df_joined.join(df_org, df_joined["OwnershipOrganisationId"] == df_org["OrganisationId"], how="left").drop("OrganisationId")

after_count = df_joined.count()

# DUPLICATE CHECK — always runs
print(f"Row count BEFORE Organisation join: {before_count:,}")
print(f"Row count AFTER Organisation join:  {after_count:,}")
print(f"STATUS: {'No duplicates' if before_count == after_count else f'DUPLICATES DETECTED ({after_count - before_count:,} extra rows)'}")

StatementMeta(, f1b3efe3-c90b-45f6-9d62-00a7d2401418, 9, Finished, Available, Finished, False)

Row count BEFORE Organisation join: 48,159,303
Row count AFTER Organisation join:  48,159,303
STATUS: No duplicates


In [8]:
# Cell 7: USD Conversion

premium_revenue_cols = [
    "ClientGrossPremium", "ClientNetPremium",
    "UWGrossPremium", "UWNetPremium",
    "ClientCommission", "ClientFee", "ClientMDI",
    "UWMDI", "UWFee",
    "GrossBrokerage", "GrossPremium",
    "WTWNetRevenue", "WTWGrossRevenue",
    "ContingentCommission",
    "ClientAdditionalCommission", "UWAdditionalCommission",
    "TPCommission", "TPAdditionalCommission", "TPFee", "TPMDI",
    "UWNetTotDeds", "Claim",
    "ClientDeduction", "TPDeduction", "UWDeduction",
    "Expenses", "CostOtherExpense", "CostCompanyExpense",
    "Adjustments", "TreatyStatement", "WriteOff",
]

for col_name in premium_revenue_cols:
    df_joined = df_joined.withColumn(
        f"{col_name}USD",
        F.round(F.coalesce(F.col(col_name), F.lit(0)) * F.coalesce(F.col("USDExchangeRate"), F.lit(0)), 2)
    )

print("USD conversion complete.")

StatementMeta(, f1b3efe3-c90b-45f6-9d62-00a7d2401418, 10, Finished, Available, Finished, False)

USD conversion complete.


In [9]:
# Cell 8: Derived columns

df_joined = df_joined.withColumn("InceptionYear", F.year(F.col("InceptionDate")))

print("Derived columns added.")

StatementMeta(, f1b3efe3-c90b-45f6-9d62-00a7d2401418, 11, Finished, Available, Finished, False)

Derived columns added.


In [10]:
# Cell 9: Select Final Columns

fact_transaction_premium = df_joined.select(
    # IDs & Keys
    "ClientUnderwriterPremiumId", "PolicyId", "PolicySectionId", "TransactionId", "SourceId",
    # Client (from Transaction table join)
    "ClientPartyId",
    # Organisation (WTW team — from Organisation table)
    "Organisation",
    # Underwriter (explicit in CUP)
    "Underwriter", "UnderwriterId", "UWGlobalPartyId", "UWRole", "UWCountry", "UWParent", "UWDescName",
    # Market Details
    "Lead", "WrittenLine", "SignedLine", "PlacementType",
    "MarketLevel1", "MarketLevel2", "MarketLevel3", "MarketLevel4", "MarketRole1",
    # Bureau/CoBroker
    "Bureau", "BureauId", "CoBroker", "CoBrokerId",
    # Dates
    "GLAccountingDate", "InceptionDate", "FirstInceptionDate", "ExpiryDate", "RenewalDate", "InceptionYear",
    # Policy
    "RefInsuranceTypeId", "RefInsuranceType",
    # Product (from Transaction table join)
    "ProductKey",
    "GlobalProductClassId", "GlobalProductClass",
    "GlobalProductLineId", "GlobalProductLine",
    "GlobalProductId", "GlobalProduct",
    # Currency
    "OrigCcyISO", "USDExchangeRate",
    # Premium & Revenue (local currency)
    "ClientGrossPremium", "ClientNetPremium",
    "UWGrossPremium", "UWNetPremium",
    "ClientCommission", "ClientFee", "ClientMDI",
    "UWMDI", "UWFee",
    "GrossBrokerage", "GrossPremium",
    "WTWNetRevenue", "WTWGrossRevenue",
    "ContingentCommission",
    "ClientAdditionalCommission", "UWAdditionalCommission",
    # Premium & Revenue (USD)
    "ClientGrossPremiumUSD", "ClientNetPremiumUSD",
    "UWGrossPremiumUSD", "UWNetPremiumUSD",
    "ClientCommissionUSD", "ClientFeeUSD", "ClientMDIUSD",
    "UWMDIUSD", "UWFeeUSD",
    "GrossBrokerageUSD", "GrossPremiumUSD",
    "WTWNetRevenueUSD", "WTWGrossRevenueUSD",
    "ContingentCommissionUSD",
    "ClientAdditionalCommissionUSD", "UWAdditionalCommissionUSD",
)

if RUN_STATS:
    row_count = fact_transaction_premium.count()
    print(f"Fact_TransactionPremium rows: {row_count:,}\n")

    print("Column NULL counts:")
    print("-" * 60)
    for col_name in fact_transaction_premium.columns:
        null_count = fact_transaction_premium.filter(F.col(col_name).isNull()).count()
        pct = null_count / row_count * 100 if row_count > 0 else 0
        print(f"  {col_name:<35} {null_count:>12,}  ({pct:5.1f}%)")
else:
    print("Final columns selected.")

StatementMeta(, f1b3efe3-c90b-45f6-9d62-00a7d2401418, 12, Finished, Available, Finished, False)

Final columns selected.


In [11]:
# Cell 10: Write to Gold Lakehouse

fact_transaction_premium.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("Gold_SalesOps_Fact_TransactionPremium")

final_count = spark.read.table("Gold_SalesOps_Fact_TransactionPremium").count()
print(f"Gold_SalesOps_Fact_TransactionPremium written: {final_count:,} rows")

StatementMeta(, f1b3efe3-c90b-45f6-9d62-00a7d2401418, 13, Finished, Available, Finished, False)

Gold_SalesOps_Fact_TransactionPremium written: 48,159,303 rows
